In [1]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join('..', '..')))

import argparse
from pprint import pp
import torch
from torch import nn
from tqdm import tqdm
import numpy as np
import json
import os
from omegaconf import OmegaConf
from torch.utils.tensorboard import SummaryWriter

from lpn.utils import load_dataset, load_config
from lpn.utils import get_model
from lpn.utils import get_loss_hparams_and_lr, get_loss
from lpn.utils import trainer
from lpn.utils import utils
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
lpn_model_config_path = os.path.join('..','models','lpn','s=0.1','model_config.json')
lpn_model_weight_path = os.path.join('..','models','lpn','s=0.1','model.pt')

lpn_model_config = load_config(lpn_model_config_path)
lpn_model = get_model(lpn_model_config)
lpn_model.load_state_dict(torch.load(lpn_model_weight_path)['model_state_dict'])
lpn_model.to(device)

init weights


LPN(
  (lin): ModuleList(
    (0): Conv2d(3, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (3): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (5): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (6): Conv2d(256, 64, kernel_size=(16, 16), stride=(1, 1), bias=False)
    (7): Linear(in_features=64, out_features=1, bias=True)
  )
  (res): ModuleList(
    (0): Conv2d(3, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): Conv2d(3, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (2): Conv2d(3, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (3): Conv2d(3, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 

In [4]:
# The NE LPN model by the forward normalization trick
ne_lpn_by_forward_model_config_path = '../models/ne_by_forward_lpn/s=0.1/model_config.json'
ne_lpn_by_forward_model_weight_path = '../models/ne_by_forward_lpn/s=0.1/model.pt'
ne_lpn_by_forward_model_config = load_config(ne_lpn_by_forward_model_config_path)
ne_lpn_by_forward_model = get_model(ne_lpn_by_forward_model_config)
ne_lpn_by_forward_model.load_state_dict(torch.load(ne_lpn_by_forward_model_weight_path)['model_state_dict'])
ne_lpn_by_forward_model.to(device)

init weights


LPN(
  (lin): ModuleList(
    (0): Conv2d(3, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (3): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (5): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (6): Conv2d(256, 64, kernel_size=(16, 16), stride=(1, 1), bias=False)
    (7): Linear(in_features=64, out_features=1, bias=True)
  )
  (res): ModuleList(
    (0): Conv2d(3, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): Conv2d(3, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (2): Conv2d(3, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (3): Conv2d(3, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 

In [5]:
# Scale-Equivariant By Design LPN

scale_eq_lpn_by_design_config_path = os.path.join('..','models','scale_equiv_lpn','s=0.1','model_config.json')
scale_eq_lpn_by_design_weight_path = os.path.join('..','models','scale_equiv_lpn','s=0.1','model.pt')
scale_eq_lpn_by_design_config = load_config(scale_eq_lpn_by_design_config_path)
scale_eq_lpn_by_design_model = get_model(scale_eq_lpn_by_design_config)
scale_eq_lpn_by_design_model.load_state_dict(torch.load(scale_eq_lpn_by_design_weight_path)['model_state_dict'])
scale_eq_lpn_by_design_model.to(device)

init weights


LPN(
  (lin): ModuleList(
    (0): Conv2d(3, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (3): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (5): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (6): Conv2d(256, 64, kernel_size=(16, 16), stride=(1, 1), bias=False)
    (7): Linear(in_features=64, out_features=1, bias=False)
  )
  (res): ModuleList(
    (0): Conv2d(3, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): Conv2d(3, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (2): Conv2d(3, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (3): Conv2d(3, 256

In [6]:
# NE-LPN by Design (with SortPool activation)

ne_by_design_lpn_model_config_path = os.path.join('..','models','ne_by_design_lpn','s=0.1','model_config.json')
ne_by_design_lpn_model_weight_path = os.path.join('..','models','ne_by_design_lpn','s=0.1','model.pt')
ne_by_design_lpn_model_config = load_config(ne_by_design_lpn_model_config_path)
ne_by_design_lpn_model = get_model(ne_by_design_lpn_model_config)
ne_by_design_lpn_model.load_state_dict(torch.load(ne_by_design_lpn_model_weight_path)['model_state_dict'])
ne_by_design_lpn_model.to(device)


init weights


LPN(
  (lin): ModuleList(
    (0): Conv2d(3, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (3): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (5): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (6): Conv2d(256, 64, kernel_size=(16, 16), stride=(1, 1), bias=False)
    (7): Linear(in_features=64, out_features=1, bias=False)
  )
  (res): ModuleList(
    (0): Conv2d(3, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): Conv2d(3, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (2): Conv2d(3, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (3): Conv2d(3, 256

In [7]:
dataset_config_path = os.path.join('..','configs','dataset.json')
dataset_config = load_config(dataset_config_path)
dataset_config['params']['root'] = '../../../data/bsds500'
test_dataset = load_dataset(dataset_config, 'test')
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=4)

dataset:  bsds500


In [8]:
noise_level = 0.1

This experiment works by analyzing denoising performance under
affine scale and shift, of form f(alpha * x + beta).
If beta = 0, as alpha decreases, the input image to denoise should become darker.
If beta = 1 - alpha, the image should become whiter as alpha decreases.

In [9]:
# beta will be either 0 or 1 - alpha, so we just need the alpha values

alpha_scales = [1.0, 0.75, 0.5, 0.25]

In [10]:
from skimage.metrics import peak_signal_noise_ratio as skimage_psnr
from skimage.metrics import structural_similarity as skimage_ssim
from collections import defaultdict

In [11]:
def evaluate_model_on_transformed_inputs(model, dataloader, alpha_scales, noise_level, mode="darken"):
    if mode not in ['darken', 'brighten']:
        raise ValueError("Mode must be either 'darken' or 'brighten'")
    model.eval()
    avg_psnr_list = []
    avg_ssim_list = []
    sd_psnr_list = []
    sd_ssim_list = []
    img_dict = defaultdict(list)

    for alpha_scale in tqdm(alpha_scales, total=len(alpha_scales)):
        beta = 0 if mode == "darken" else 1 - alpha_scale
        psnr_list = []
        ssim_list = []

        for i, data in tqdm(enumerate(dataloader), total=len(dataloader)):
            img = data['image'].to(device)
            noise_tensor = torch.randn_like(img) * noise_level
            transformed_img = alpha_scale * img + beta
            noisy_img = transformed_img + noise_tensor

            with torch.no_grad():
                denoised_img = model(noisy_img)

            img_np = img.squeeze(0).permute(1,2,0).cpu().detach().numpy()
            transformed_np = transformed_img.squeeze(0).permute(1,2,0).cpu().detach().numpy()
            denoised_np = denoised_img.squeeze(0).permute(1,2,0).cpu().detach().numpy()
            img_dict[alpha_scale].append((img_np, transformed_np, denoised_np))

            psnr = skimage_psnr(transformed_np, denoised_np, data_range=1.0)
            ssim = skimage_ssim(transformed_np, denoised_np, data_range=1.0, channel_axis=-1)

            psnr_list.append(psnr)
            ssim_list.append(ssim)
        avg_psnr = np.mean(psnr_list)
        avg_ssim = np.mean(ssim_list)
        sd_psnr = np.std(psnr_list)
        sd_ssim = np.std(ssim_list)
        avg_psnr_list.append(avg_psnr)
        avg_ssim_list.append(avg_ssim)
        sd_psnr_list.append(sd_psnr)
        sd_ssim_list.append(sd_ssim)
        print(f"Alpha Scale: {alpha_scale}, Avg PSNR: {avg_psnr:.2f} +/- {sd_psnr:.2f}, Avg SSIM: {avg_ssim:.4f} +/- {sd_ssim:.4f}")
    return avg_psnr_list, sd_psnr_list, avg_ssim_list, sd_ssim_list, img_dict

In [12]:
import imageio.v2 as imageio

def save_images(img_dict, save_dir="../results/affine_denoised_images"):
    os.makedirs(save_dir, exist_ok=True)

    for scale, img_pairs in tqdm(img_dict.items(), total=len(img_dict), desc="Saving images"):
        scale_dir = os.path.join(save_dir, f"scale_{scale}")
        os.makedirs(scale_dir, exist_ok=True)

        for idx, (gt, transformed, denoised) in enumerate(img_pairs):
            gt_path = os.path.join(scale_dir, f"gt_{idx}.png")
            transformed_path = os.path.join(scale_dir, f"transformed_{idx}.png")
            denoised_path = os.path.join(scale_dir, f"denoised_{idx}.png")

            gt_uint8 = (np.clip(gt, 0, 1) * 255).astype(np.uint8)
            transformed_uint8 = (np.clip(transformed, 0, 1) * 255).astype(np.uint8)
            denoised_uint8 = (np.clip(denoised, 0, 1) * 255).astype(np.uint8)

            imageio.imwrite(gt_path, gt_uint8)
            imageio.imwrite(transformed_path, transformed_uint8)
            imageio.imwrite(denoised_path, denoised_uint8)


In [13]:
save_dir = '../affine_denoising_results' 
os.makedirs(save_dir, exist_ok=True)

In [14]:
lpn_darken_psnr, lpn_darken_ssim, lpn_darken_sd_psnr, lpn_darken_sd_ssim, lpn_darken_img_dict = evaluate_model_on_transformed_inputs(lpn_model, test_dataloader, alpha_scales, noise_level, mode="darken")
np.savez(
    os.path.join(save_dir, 'lpn_darken_results.npz'),
    alpha_scales=alpha_scales,
    psnr=np.array(lpn_darken_psnr, dtype=float),
    ssim=np.array(lpn_darken_ssim, dtype=float),
    sd_psnr=np.array(lpn_darken_sd_psnr, dtype=float),
    sd_ssim=np.array(lpn_darken_sd_ssim, dtype=float),
    model_name="LPN"
)
save_images(lpn_darken_img_dict, save_dir=os.path.join(save_dir, 'darken', 'lpn'))

 25%|██▌       | 1/4 [00:04<00:13,  4.47s/it]

Alpha Scale: 1.0, Avg PSNR: 29.55 +/- 1.60, Avg SSIM: 0.8599 +/- 0.0381


 50%|█████     | 2/4 [00:07<00:07,  3.52s/it]

Alpha Scale: 0.75, Avg PSNR: 30.50 +/- 1.62, Avg SSIM: 0.8456 +/- 0.0385


 75%|███████▌  | 3/4 [00:10<00:03,  3.38s/it]

Alpha Scale: 0.5, Avg PSNR: 31.86 +/- 1.62, Avg SSIM: 0.8325 +/- 0.0411


100%|██████████| 4/4 [00:13<00:00,  3.38s/it]


Alpha Scale: 0.25, Avg PSNR: 34.38 +/- 1.61, Avg SSIM: 0.8381 +/- 0.0433


Saving images: 100%|██████████| 4/4 [00:20<00:00,  5.23s/it]


In [15]:
lpn_brighten_psnr, lpn_brighten_ssim, lpn_brighten_sd_psnr, lpn_brighten_sd_ssim, lpn_brighten_img_dict = evaluate_model_on_transformed_inputs(lpn_model, test_dataloader, alpha_scales, noise_level, mode="brighten")
np.savez(
    os.path.join(save_dir, 'lpn_brighten_results.npz'),
    alpha_scales=alpha_scales,
    psnr=np.array(lpn_brighten_psnr, dtype=float),
    ssim=np.array(lpn_brighten_ssim, dtype=float),
    sd_psnr=np.array(lpn_brighten_sd_psnr, dtype=float),
    sd_ssim=np.array(lpn_brighten_sd_ssim, dtype=float),
    model_name="LPN"
)
save_images(lpn_brighten_img_dict, save_dir=os.path.join(save_dir, 'brighten', 'lpn'))

 25%|██▌       | 1/4 [00:02<00:08,  2.81s/it]

Alpha Scale: 1.0, Avg PSNR: 29.55 +/- 1.59, Avg SSIM: 0.8598 +/- 0.0380


 50%|█████     | 2/4 [00:05<00:05,  2.62s/it]

Alpha Scale: 0.75, Avg PSNR: 30.44 +/- 1.58, Avg SSIM: 0.8451 +/- 0.0375


 75%|███████▌  | 3/4 [00:08<00:02,  2.73s/it]

Alpha Scale: 0.5, Avg PSNR: 31.60 +/- 1.50, Avg SSIM: 0.8274 +/- 0.0379


100%|██████████| 4/4 [00:11<00:00,  2.78s/it]


Alpha Scale: 0.25, Avg PSNR: 33.47 +/- 1.15, Avg SSIM: 0.8198 +/- 0.0316


Saving images: 100%|██████████| 4/4 [00:38<00:00,  9.53s/it]


In [22]:
ne_by_forward_darken_psnr, ne_by_forward_darken_ssim, ne_by_forward_darken_sd_psnr, ne_by_forward_darken_sd_ssim, ne_by_forward_darken_img_dict = evaluate_model_on_transformed_inputs(ne_lpn_by_forward_model, test_dataloader, alpha_scales, noise_level, mode="darken")
np.savez(
    os.path.join(save_dir, 'ne_by_forward_darken_results.npz'),
    alpha_scales=alpha_scales,
    psnr=np.array(ne_by_forward_darken_psnr, dtype=float),
    ssim=np.array(ne_by_forward_darken_ssim, dtype=float),
    sd_psnr=np.array(ne_by_forward_darken_sd_psnr, dtype=float),
    sd_ssim=np.array(ne_by_forward_darken_sd_ssim, dtype=float),
    model_name="NE-LPN (By Normalization Trick)"
)
save_images(ne_by_forward_darken_img_dict, save_dir=os.path.join(save_dir, 'darken', 'ne_by_forward'))

 25%|██▌       | 1/4 [00:02<00:08,  2.81s/it]

Alpha Scale: 1.0, Avg PSNR: 29.10 +/- 1.64, Avg SSIM: 0.8443 +/- 0.0387


 50%|█████     | 2/4 [00:05<00:05,  2.92s/it]

Alpha Scale: 0.75, Avg PSNR: 30.12 +/- 1.41, Avg SSIM: 0.8319 +/- 0.0407


 75%|███████▌  | 3/4 [00:08<00:02,  2.77s/it]

Alpha Scale: 0.5, Avg PSNR: 30.82 +/- 1.12, Avg SSIM: 0.7923 +/- 0.0453


100%|██████████| 4/4 [00:11<00:00,  2.89s/it]


Alpha Scale: 0.25, Avg PSNR: 31.12 +/- 0.65, Avg SSIM: 0.6998 +/- 0.0342


Saving images: 100%|██████████| 4/4 [00:19<00:00,  4.79s/it]


In [17]:
ne_by_forward_brighten_psnr, ne_by_forward_brighten_ssim, ne_by_forward_brighten_sd_psnr, ne_by_forward_brighten_sd_ssim, ne_by_forward_brighten_img_dict = evaluate_model_on_transformed_inputs(ne_lpn_by_forward_model, test_dataloader, alpha_scales, noise_level, mode="brighten")
np.savez(
    os.path.join(save_dir, 'ne_by_forward_brighten_results.npz'),
    alpha_scales=alpha_scales,
    psnr=np.array(ne_by_forward_brighten_psnr, dtype=float),
    ssim=np.array(ne_by_forward_brighten_ssim, dtype=float),
    sd_psnr=np.array(ne_by_forward_brighten_sd_psnr, dtype=float),
    sd_ssim=np.array(ne_by_forward_brighten_sd_ssim, dtype=float),
    model_name="NE-LPN (By Normalization Trick)"
)
save_images(ne_by_forward_brighten_img_dict, save_dir=os.path.join(save_dir, 'brighten', 'ne_by_forward'))

 25%|██▌       | 1/4 [00:02<00:08,  2.73s/it]

Alpha Scale: 1.0, Avg PSNR: 29.09 +/- 1.63, Avg SSIM: 0.8441 +/- 0.0388


 50%|█████     | 2/4 [00:06<00:06,  3.06s/it]

Alpha Scale: 0.75, Avg PSNR: 30.12 +/- 1.40, Avg SSIM: 0.8346 +/- 0.0411


 75%|███████▌  | 3/4 [00:09<00:03,  3.17s/it]

Alpha Scale: 0.5, Avg PSNR: 30.82 +/- 1.13, Avg SSIM: 0.7967 +/- 0.0449


100%|██████████| 4/4 [00:12<00:00,  3.13s/it]


Alpha Scale: 0.25, Avg PSNR: 31.11 +/- 0.67, Avg SSIM: 0.7117 +/- 0.0320


Saving images: 100%|██████████| 4/4 [00:27<00:00,  6.77s/it]


In [18]:
scale_by_design_darken_psnr, scale_by_design_darken_ssim, scale_by_design_darken_sd_psnr, scale_by_design_darken_sd_ssim, scale_by_design_darken_img_dict = evaluate_model_on_transformed_inputs(scale_eq_lpn_by_design_model, test_dataloader, alpha_scales, noise_level, mode="darken")
np.savez(
    os.path.join(save_dir, 'scale_by_design_darken_results.npz'),
    alpha_scales=alpha_scales,
    psnr=np.array(scale_by_design_darken_psnr, dtype=float),
    ssim=np.array(scale_by_design_darken_ssim, dtype=float),
    sd_psnr=np.array(scale_by_design_darken_sd_psnr, dtype=float),
    sd_ssim=np.array(scale_by_design_darken_sd_ssim, dtype=float),
    model_name="Scale-Equivariant LPN (By Design)"
)
save_images(scale_by_design_darken_img_dict, save_dir=os.path.join(save_dir, 'darken', 'scale_by_design'))

 25%|██▌       | 1/4 [00:03<00:10,  3.39s/it]

Alpha Scale: 1.0, Avg PSNR: 17.14 +/- 2.50, Avg SSIM: 0.5041 +/- 0.0707


 50%|█████     | 2/4 [00:06<00:06,  3.32s/it]

Alpha Scale: 0.75, Avg PSNR: 19.68 +/- 2.46, Avg SSIM: 0.5387 +/- 0.0714


 75%|███████▌  | 3/4 [00:09<00:03,  3.30s/it]

Alpha Scale: 0.5, Avg PSNR: 23.11 +/- 2.24, Avg SSIM: 0.5718 +/- 0.0680


100%|██████████| 4/4 [00:13<00:00,  3.33s/it]


Alpha Scale: 0.25, Avg PSNR: 27.43 +/- 1.26, Avg SSIM: 0.5547 +/- 0.0470


Saving images: 100%|██████████| 4/4 [00:22<00:00,  5.73s/it]


In [19]:
scale_by_design_brighten_psnr, scale_by_design_brighten_ssim, scale_by_design_brighten_sd_psnr, scale_by_design_brighten_sd_ssim, scale_by_design_brighten_img_dict = evaluate_model_on_transformed_inputs(scale_eq_lpn_by_design_model, test_dataloader, alpha_scales, noise_level, mode="brighten")
np.savez(
    os.path.join(save_dir, 'scale_by_design_brighten_results.npz'),
    alpha_scales=alpha_scales,
    psnr=np.array(scale_by_design_brighten_psnr, dtype=float),
    ssim=np.array(scale_by_design_brighten_ssim, dtype=float),
    sd_psnr=np.array(scale_by_design_brighten_sd_psnr, dtype=float),
    sd_ssim=np.array(scale_by_design_brighten_sd_ssim, dtype=float),
    model_name="Scale-Equivariant LPN (By Design)"
)
save_images(scale_by_design_brighten_img_dict, save_dir=os.path.join(save_dir, 'brighten', 'scale_by_design'))

 25%|██▌       | 1/4 [00:03<00:09,  3.15s/it]

Alpha Scale: 1.0, Avg PSNR: 17.14 +/- 2.50, Avg SSIM: 0.5042 +/- 0.0708


 50%|█████     | 2/4 [00:06<00:06,  3.17s/it]

Alpha Scale: 0.75, Avg PSNR: 18.25 +/- 2.28, Avg SSIM: 0.5006 +/- 0.0739


 75%|███████▌  | 3/4 [00:09<00:03,  3.31s/it]

Alpha Scale: 0.5, Avg PSNR: 19.73 +/- 1.66, Avg SSIM: 0.5205 +/- 0.0802


100%|██████████| 4/4 [00:13<00:00,  3.36s/it]


Alpha Scale: 0.25, Avg PSNR: 20.68 +/- 0.81, Avg SSIM: 0.6099 +/- 0.0771


Saving images: 100%|██████████| 4/4 [00:33<00:00,  8.45s/it]


In [20]:
ne_by_design_darken_psnr, ne_by_design_darken_ssim, ne_by_design_darken_sd_psnr, ne_by_design_darken_sd_ssim, ne_by_design_darken_img_dict = evaluate_model_on_transformed_inputs(ne_by_design_lpn_model, test_dataloader, alpha_scales, noise_level, mode="darken")
np.savez(
    os.path.join(save_dir, 'ne_by_design_darken_results.npz'),
    alpha_scales=alpha_scales,
    psnr=np.array(ne_by_design_darken_psnr, dtype=float),
    ssim=np.array(ne_by_design_darken_ssim, dtype=float),
    sd_psnr=np.array(ne_by_design_darken_sd_psnr, dtype=float),
    sd_ssim=np.array(ne_by_design_darken_sd_ssim, dtype=float),
    model_name="NE-LPN (By Design)"
)
save_images(ne_by_design_darken_img_dict, save_dir=os.path.join(save_dir, 'darken', 'ne_by_design'))

 25%|██▌       | 1/4 [00:03<00:09,  3.30s/it]

Alpha Scale: 1.0, Avg PSNR: 23.55 +/- 1.80, Avg SSIM: 0.7268 +/- 0.0589


 50%|█████     | 2/4 [00:06<00:06,  3.41s/it]

Alpha Scale: 0.75, Avg PSNR: 25.69 +/- 1.60, Avg SSIM: 0.7269 +/- 0.0572


 75%|███████▌  | 3/4 [00:10<00:03,  3.57s/it]

Alpha Scale: 0.5, Avg PSNR: 28.20 +/- 1.21, Avg SSIM: 0.7140 +/- 0.0552


100%|██████████| 4/4 [00:14<00:00,  3.55s/it]


Alpha Scale: 0.25, Avg PSNR: 30.65 +/- 0.52, Avg SSIM: 0.6664 +/- 0.0403


Saving images: 100%|██████████| 4/4 [00:30<00:00,  7.60s/it]


In [21]:
ne_by_design_brighten_psnr, ne_by_design_brighten_ssim, ne_by_design_brighten_sd_psnr, ne_by_design_brighten_sd_ssim, ne_by_design_brighten_img_dict = evaluate_model_on_transformed_inputs(ne_by_design_lpn_model, test_dataloader, alpha_scales, noise_level, mode="brighten")
np.savez(
    os.path.join(save_dir, 'ne_by_design_brighten_results.npz'),
    alpha_scales=alpha_scales,
    psnr=np.array(ne_by_design_brighten_psnr, dtype=float),
    ssim=np.array(ne_by_design_brighten_ssim, dtype=float),
    sd_psnr=np.array(ne_by_design_brighten_sd_psnr, dtype=float),
    sd_ssim=np.array(ne_by_design_brighten_sd_ssim, dtype=float),
    model_name="NE-LPN (By Design)"
)
save_images(ne_by_design_brighten_img_dict, save_dir=os.path.join(save_dir, 'brighten', 'ne_by_design'))

 25%|██▌       | 1/4 [00:03<00:10,  3.38s/it]

Alpha Scale: 1.0, Avg PSNR: 23.55 +/- 1.80, Avg SSIM: 0.7271 +/- 0.0585


 50%|█████     | 2/4 [00:07<00:07,  3.54s/it]

Alpha Scale: 0.75, Avg PSNR: 25.69 +/- 1.60, Avg SSIM: 0.7370 +/- 0.0553


 75%|███████▌  | 3/4 [00:10<00:03,  3.53s/it]

Alpha Scale: 0.5, Avg PSNR: 28.19 +/- 1.21, Avg SSIM: 0.7244 +/- 0.0548


100%|██████████| 4/4 [00:14<00:00,  3.50s/it]


Alpha Scale: 0.25, Avg PSNR: 30.65 +/- 0.53, Avg SSIM: 0.6790 +/- 0.0384


Saving images: 100%|██████████| 4/4 [00:27<00:00,  7.00s/it]
